# Notebook 06 — Full Model and Leave-One-Out Ablation Matrix

**Summary:** Closes the loop — assembles the full proposed model and runs the complete
leave-one-out ablation sweep. Produces the master results table for the thesis.

**Full model components:**
- C1: R-Blur foveal transform (`src/foveation.py`)
- C2: Trace-based metameric periphery (Watson-SNR grounded)
- C3: VOneBlock Poisson noise (`src/overrides.py`)
- C4: ResNet-50 ventral back-end (`src/models.py`)
- C5: Multi-glance + confidence halting (`src/it_feedback.py`)

**Outputs:** `results/06_ablation_table.json` · `results/06_ablation_table.png` ·
`results/06_efficiency_curve.png`

---

## Mathematics

**Marginal contribution** of component $C_k$:

$$
\Delta_k = \mathrm{metric}(\text{full model}) - \mathrm{metric}(\text{full} \setminus C_k)
$$

**EOT adversarial gradient** (for stochastic C3):

$$
\bar g = \frac{1}{N}\sum_{i=1}^N \nabla_x \ell(f(T_i(x)),\, y), \quad N\ge10
$$

**Efficiency curve:** accuracy $A$ vs average glances $G$:

$$
\eta = \frac{A}{G} \quad \text{(accuracy-per-glance efficiency)}
$$

> **Smoke-test scope:** All models use random weights; accuracy numbers are not
> meaningful. The notebook demonstrates the full pipeline and ablation machinery.


In [ ]:
# ── Environment ─────────────────────────────────────────────────────────────
import sys, os, warnings, json
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/tez_foveated_vision"
else:
    _here = os.getcwd()
    for _d in [_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))]:
        if os.path.isdir(os.path.join(_d, "src")):
            PROJECT_ROOT = _d; break
    else: PROJECT_ROOT = _here

os.chdir(PROJECT_ROOT)
for _p in [PROJECT_ROOT,
           os.path.join(PROJECT_ROOT, "external", "vonenet"),
           os.path.join(PROJECT_ROOT, "external", "CORnet")]:
    if _p not in sys.path: sys.path.insert(0, _p)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import common, datasets, models
from src.foveation import (SNRProfile, build_foveated_transform,
                            apply_rblur, FoveatedTransform, TraceBasedPeriphery)
from src.overrides  import apply_vone_block_input_gradient_fix, set_v1_noise_mode
from src.it_feedback import FixationGrid, MultiGlanceFoveatedModel
from src.eval_harness import pgd_attack

CFG = {
    "seed": 42, "smoke_test": True,
    # ImageNet-100 (100-class ImageNet-1K subset) at VOneNet's native 224px/ppd=28
    # scale -- switched from CIFAR-10 (32px/ppd=4) because a frozen, ImageNet-
    # pretrained VOneNet/CORnet-S backbone fed 32px input produces badly degraded
    # features (same 8deg-total field-of-view convention either way: 224/28 ==
    # 32/4 == 8, so the beta-sweep's alpha(r) numbers below are unaffected).
    "image_size": 224, "n_classes": 100, "ppd": 28.0,
    "fovea_deg": 1.5, "transition_deg": 0.5,
    "rblur_sigma0": 0.5, "rblur_slope": 2.0,
    "snr0_db": 30.0, "beta": 2.0, "patch_size": 32,
    "max_glances": 4, "halt_threshold": 0.75, "margin": 0.25,
    # Beta sweep for the "full" model (VOneNet + trace periphery + multi-glance
    # combined) -- beta=1-3 alone barely change alpha(r) within an 8deg field of
    # view (see 02_foveation_rblur_and_periphery.ipynb's beta-sweep figure).
    "beta_sweep": [1.0, 2.0, 3.0, 5.0, 8.0],
    # Training
    "n_epochs_smoke": 3, "n_batches_smoke": 20,
    "n_epochs_full": 20, "n_batches_full": None,
    "batch_size": 16, "lr": 1e-3,   # smaller batch: 224px + multi-glance is heavier than 32px CIFAR-10
    "data_dir": os.path.join(PROJECT_ROOT, "data"),
    # Adversarial
    "eps": 4/255, "pgd_steps": 5, "pgd_alpha": 1/255, "eot_samples": 4,
    # ImageNet-C (filtered to our 100 classes) evaluation, for all leave-one-out
    # conditions and all beta-sweep values
    "n_batches_imagenet_c_smoke": 3,
    "result_dir": os.path.join(PROJECT_ROOT, "results"),
    "ckpt_dir":   os.path.join(PROJECT_ROOT, "checkpoints"),
}

common.set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["result_dir"], exist_ok=True)
apply_vone_block_input_gradient_fix()
print(f"Device: {device} | Smoke test: {CFG['smoke_test']}")


## Assembling the Full Model

The full model pipeline:

```
x_raw  (raw [0,1] pixels)
  ↓  FoveatedTransform (C1: R-Blur  +  C2: trace-based periphery)
  ↓  NormalizedModel(VOneResNet-50, C3: Poisson noise, C4: ResNet-50 back-end)
  ↓  MultiGlanceFoveatedModel (C5: multi-glance + confidence halting)
  →  logits
```

For leave-one-out: each ablation removes exactly one component and keeps all others.


In [ ]:
# ── Build the full model and all ablation variants ───────────────────────

PRETRAINED = not CFG["smoke_test"]
S = CFG["image_size"]
snr = SNRProfile(snr0_db=CFG["snr0_db"], beta=CFG["beta"], ppd=CFG["ppd"])
grid = FixationGrid(2, 2, CFG["margin"])
fixations = grid.get_fixations()

MU  = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class FullPipeline(nn.Module):
    """Full foveated model: fov_transform -> backbone -> (optional multi-glance).

    Accepts raw [0,1] pixel tensors. Applies foveation then backbone normalisation.
    """
    def __init__(self, fov_transform, backbone, norm, fixations_list=None,
                 halt_threshold=0.75, sigma0=0.5, slope=1.5, ppd=4.0,
                 multi_glance=False):
        super().__init__()
        self.fov = fov_transform     # None -> no foveation
        self.backbone = models.NormalizedModel(backbone, norm)
        self.multi_glance = multi_glance
        self.fixations = fixations_list or [(0.5, 0.5)]
        self.halt_threshold = halt_threshold
        self.sigma0 = sigma0; self.slope = slope; self.ppd = ppd

    def apply_fov(self, x_raw, fixation_yx=None):
        if self.fov is None:
            return x_raw
        # Temporarily override fixation in fov transform's periphery modules
        if fixation_yx is not None and hasattr(self.fov, 'fixation_yx'):
            self.fov.fixation_yx = fixation_yx
            if self.fov.periphery is not None:
                self.fov.periphery.fixation_yx = fixation_yx
        return torch.stack([self.fov(x_raw[i]) for i in range(x_raw.size(0))])

    def forward(self, x_raw, return_glance_count=False):
        B = x_raw.size(0)
        if not self.multi_glance:
            fov_x = self.apply_fov(x_raw)
            logits = self.backbone(fov_x)
            if return_glance_count:
                return logits, torch.ones(B, dtype=torch.long, device=x_raw.device)
            return logits

        logit_sum = None
        halted = torch.zeros(B, dtype=torch.bool, device=x_raw.device)
        gc = torch.ones(B, dtype=torch.long, device=x_raw.device)
        for t, fix in enumerate(self.fixations):
            fov_x = self.apply_fov(x_raw, fix)
            z = self.backbone(fov_x)
            logit_sum = z if logit_sum is None else logit_sum + z
            if not self.training and t < len(self.fixations) - 1:
                conf = F.softmax(logit_sum / (t+1), dim=1).max(1).values
                new_halt = (conf >= self.halt_threshold) & (~halted)
                gc += (~halted & ~new_halt).long()
                halted |= new_halt
                if halted.all(): break
        final = logit_sum / len(self.fixations)
        return (final, gc) if return_glance_count else final


def build_variant(label, PRETRAINED, snr, fixations, CFG):
    """Build one ablation variant. Returns FullPipeline."""
    # Default: full model
    use_fov    = True   # C1
    use_trace  = True   # C2
    use_vnoise = True   # C3
    use_cornet = False  # C4 variant (True = CORnet-S instead of ResNet-50)
    use_multi  = True   # C5

    if   label == "full":         pass
    elif label == "no_C1_fov":    use_fov    = False
    elif label == "no_C2_trace":  use_trace  = False
    elif label == "no_C3_noise":  use_vnoise = False
    elif label == "no_C4_resnet": use_cornet = True
    elif label == "no_C5_multi":  use_multi  = False

    # Back-end
    if use_cornet:
        backbone, norm = models.build_cornet_s(pretrained=PRETRAINED)
    else:
        backbone, norm = models.build_voneresnet50(pretrained=PRETRAINED)
        set_v1_noise_mode(backbone, mode="neuronal" if use_vnoise else None)

    backbone.eval()

    # Foveation transform
    if not use_fov:
        ft = None
    elif not use_trace:
        ft = build_foveated_transform("blur", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5),
                                       CFG["rblur_sigma0"], CFG["rblur_slope"])
    else:
        ft = build_foveated_transform("trace", snr, CFG["patch_size"],
                                       CFG["fovea_deg"], CFG["transition_deg"],
                                       CFG["ppd"], (0.5, 0.5))

    return FullPipeline(ft, backbone, norm,
                        fixations_list=fixations if use_multi else [(0.5, 0.5)],
                        halt_threshold=CFG["halt_threshold"],
                        sigma0=CFG["rblur_sigma0"],
                        slope=CFG["rblur_slope"],
                        ppd=CFG["ppd"],
                        multi_glance=use_multi)


ABLATION_LABELS = ["full", "no_C1_fov", "no_C2_trace",
                    "no_C3_noise", "no_C4_resnet", "no_C5_multi"]

print("Building ablation variants...")
ablation_models = {}
for lbl in ABLATION_LABELS:
    ablation_models[lbl] = build_variant(lbl, PRETRAINED, snr, fixations, CFG).to(device)
    n_params = sum(p.numel() for p in ablation_models[lbl].parameters()) / 1e6
    print(f"  {lbl:20s}  {n_params:.1f}M params")

In [ ]:
# ── Fetch ImageNet-100 onto the runtime's local disk (Kaggle: ambityga/imagenet100) ──
# Google Drive's FUSE mount is slow and unreliable for ~130k small JPEG files --
# `shutil.move`-ing the reorganised dataset directly onto a Drive path caused
# intermittent FileNotFoundErrors mid-training (files listed by os.listdir but
# not actually readable yet -- a sync-lag symptom). Fix: do the download +
# Kaggle-layout reorganisation entirely on local disk (/content, fast + reliable
# SSD), then persist ONE archive file to Drive (a single large-file transfer is
# fine; thousands of small ones are not) so later sessions just copy+extract
# that one file instead of re-downloading ~15 GB from Kaggle every time.
#
# Needs a Kaggle API token. On Colab, add it via the Secrets panel (key icon,
# left sidebar) as KAGGLE_USERNAME / KAGGLE_KEY -- never paste a token directly
# into a cell: it would be saved into the notebook file (and into git history
# if committed), permanently exposing it to anyone with repo access.

CFG["imagenet100_data_dir"] = "/content" if IN_COLAB else CFG["data_dir"]
IMAGENET100_LOCAL_ROOT  = os.path.join(CFG["imagenet100_data_dir"], "imagenet100")
IMAGENET100_TRAIN       = os.path.join(IMAGENET100_LOCAL_ROOT, "train")
IMAGENET100_VAL         = os.path.join(IMAGENET100_LOCAL_ROOT, "val")
IMAGENET100_ARCHIVE     = os.path.join(CFG["data_dir"], "imagenet100_prepared.tar")  # one file, lives on Drive

if os.path.isdir(IMAGENET100_TRAIN) and os.path.isdir(IMAGENET100_VAL) \
        and os.listdir(IMAGENET100_TRAIN) and os.listdir(IMAGENET100_VAL):
    print(f"ImageNet-100 already on local disk: "
          f"{len(os.listdir(IMAGENET100_TRAIN))} train classes, "
          f"{len(os.listdir(IMAGENET100_VAL))} val classes.")
elif CFG["smoke_test"]:
    print("smoke_test=True: skipping ImageNet-100 fetch "
          "(datasets.get_imagenet100 will use a synthetic pipeline check instead).")
elif os.path.exists(IMAGENET100_ARCHIVE):
    import tarfile
    print(f"Extracting cached archive from Drive: {IMAGENET100_ARCHIVE} "
          f"({os.path.getsize(IMAGENET100_ARCHIVE) / 1e9:.1f} GB)...")
    os.makedirs(IMAGENET100_LOCAL_ROOT, exist_ok=True)
    with tarfile.open(IMAGENET100_ARCHIVE) as tf:
        tf.extractall(IMAGENET100_LOCAL_ROOT)
    print(f"Extracted: {len(os.listdir(IMAGENET100_TRAIN))} train classes, "
          f"{len(os.listdir(IMAGENET100_VAL))} val classes.")
else:
    import getpass, shutil, subprocess, tarfile

    try:
        from google.colab import userdata
        kaggle_username = userdata.get("KAGGLE_USERNAME")
        kaggle_key      = userdata.get("KAGGLE_KEY")
    except Exception:
        print("Colab secrets (KAGGLE_USERNAME/KAGGLE_KEY) not found -- enter manually.")
        kaggle_username = input("Kaggle username: ")
        kaggle_key      = getpass.getpass("Kaggle API key: ")

    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
    del kaggle_username, kaggle_key   # don't keep the token in a live variable longer than needed

    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

    raw_dir = "/content/imagenet100_raw" if IN_COLAB else os.path.join(CFG["data_dir"], "_imagenet100_raw")
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(IMAGENET100_TRAIN, exist_ok=True)
    os.makedirs(IMAGENET100_VAL, exist_ok=True)

    print("Downloading ambityga/imagenet100 from Kaggle (~15 GB, one-time, to local disk)...")
    subprocess.run(["kaggle", "datasets", "download", "-d", "ambityga/imagenet100",
                     "-p", raw_dir, "--unzip"], check=True)

    # train.X1..train.X4 hold disjoint subsets of the 100 class folders -- merge
    # them into a single train/. val.X -> val/. All still on local disk.
    for part in ["train.X1", "train.X2", "train.X3", "train.X4"]:
        part_dir = os.path.join(raw_dir, part)
        if not os.path.isdir(part_dir):
            continue
        for cls in os.listdir(part_dir):
            src, dst = os.path.join(part_dir, cls), os.path.join(IMAGENET100_TRAIN, cls)
            if os.path.isdir(src) and not os.path.exists(dst):
                shutil.move(src, dst)

    val_src = os.path.join(raw_dir, "val.X")
    if os.path.isdir(val_src):
        for cls in os.listdir(val_src):
            dst = os.path.join(IMAGENET100_VAL, cls)
            if not os.path.exists(dst):
                shutil.move(os.path.join(val_src, cls), dst)

    shutil.rmtree(raw_dir, ignore_errors=True)
    print(f"ImageNet-100 ready on local disk: {len(os.listdir(IMAGENET100_TRAIN))} train classes, "
          f"{len(os.listdir(IMAGENET100_VAL))} val classes.")

    # Persist ONE archive to Drive so future sessions skip the Kaggle download.
    print(f"Archiving to Drive for future sessions: {IMAGENET100_ARCHIVE} ...")
    os.makedirs(os.path.dirname(IMAGENET100_ARCHIVE), exist_ok=True)
    with tarfile.open(IMAGENET100_ARCHIVE, "w") as tf:
        tf.add(IMAGENET100_TRAIN, arcname="train")
        tf.add(IMAGENET100_VAL, arcname="val")
    print("Archived.")

In [ ]:
# ── Real ImageNet-100 + training / evaluation ─────────────────────────────
# Switched from CIFAR-10: a frozen, ImageNet-pretrained VOneNet/CORnet-S backbone
# fed 32px CIFAR-10 input produces badly degraded features (the backbone's conv
# stack is tuned for 224px). ImageNet-100 matches VOneNet's native scale.
# Reads from CFG["imagenet100_data_dir"] (set in the fetch cell above) -- the
# runtime's *local* disk on Colab, not the Drive-mounted CFG["data_dir"], since
# Drive's FUSE mount is unreliable for ~130k small JPEG files (see that cell's
# comment for the FileNotFoundError this caused).

train_ds = datasets.get_imagenet100(CFG["imagenet100_data_dir"], split="train", normalization="imagenet",
                                     image_size=CFG["image_size"], smoke_test=CFG["smoke_test"])
val_ds   = datasets.get_imagenet100(CFG["imagenet100_data_dir"], split="val", normalization="imagenet",
                                     image_size=CFG["image_size"], smoke_test=CFG["smoke_test"])
_num_workers = 2 if not CFG["smoke_test"] else 0
train_ld = torch.utils.data.DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                                        num_workers=_num_workers)
val_ld   = torch.utils.data.DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                                        num_workers=_num_workers)


class AblationTrainer(nn.Module):
    """Frozen FullPipeline backbone + a trainable linear head mapping its own output
    dimension (e.g. 1000 for an ImageNet-pretrained VOneResNet-50/CORnet-S backbone)
    down to CFG['n_classes'] (ImageNet-100) -- a fast linear probe, matching
    03_v1_block.ipynb's methodology, rather than full end-to-end fine-tuning (that is
    what 07_cifar10_and_cifar10c_evaluation.ipynb already does for the 5 core CIFAR-10
    conditions; this notebook's job is the full leave-one-out comparison, cheaply).
    """
    def __init__(self, pipeline, n_classes, image_size=32):
        super().__init__()
        self.pipeline = pipeline
        # Detect output dimension from a dummy forward pass
        with torch.no_grad():
            _dummy_out = pipeline(torch.rand(1, 3, image_size, image_size).to(
                next(pipeline.parameters()).device))
        out_dim = _dummy_out.shape[-1]   # e.g. 1000 for ImageNet backbone
        self.head = nn.Linear(out_dim, n_classes)

    def forward(self, x_raw, return_glance_count=False):
        out = self.pipeline(x_raw, return_glance_count=return_glance_count)
        if return_glance_count:
            logits, gc = out
            return self.head(logits), gc
        return self.head(out)


def train_ablation(pipeline, n_classes, train_loader, val_loader, n_batches, n_epochs,
                    ckpt_path, n_batches_val=None):
    """Train only `trainer.head` (pipeline stays frozen), with per-epoch loss/accuracy
    printing and checkpoint/resume support -- each of the 11 conditions in this
    notebook (6 leave-one-out + 5 beta sweep) trains independently and can resume
    mid-way after a Colab session drop, per this project's "resume-from-checkpoint
    is mandatory" rule (docs/technical-guide.md, matches
    07_cifar10_and_cifar10c_evaluation.ipynb's pattern). Re-running this cell after
    a disconnect skips conditions whose checkpoint already reached n_epochs, and
    resumes any that stopped partway -- no wasted retraining.
    """
    trainer = AblationTrainer(pipeline, n_classes, CFG["image_size"]).to(device)
    opt = torch.optim.Adam(trainer.head.parameters(), lr=CFG["lr"])

    ckpt = common.load_checkpoint(ckpt_path, map_location=device)
    if ckpt is not None:
        trainer.head.load_state_dict(ckpt["head_state"])
        opt.load_state_dict(ckpt["opt_state"])
        start_epoch = ckpt["epoch"] + 1
        history = ckpt["history"]
        print(f"    resumed from epoch {start_epoch}")
    else:
        start_epoch = 0
        history = []

    if start_epoch >= n_epochs:
        print(f"    training already complete (epoch {start_epoch}/{n_epochs})")
        return trainer, history

    for epoch in range(start_epoch, n_epochs):
        trainer.train()
        correct = total = total_loss = 0
        for bi, (xb, yb) in enumerate(train_loader):
            if n_batches is not None and bi >= n_batches: break
            xb, yb = xb.to(device), yb.to(device)
            x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
            logits = trainer(x_raw)
            loss = F.cross_entropy(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            total   += xb.size(0)
        tr_loss = total_loss / max(total, 1)
        tr_acc  = correct / max(total, 1)

        val_acc, _ = eval_pipeline(trainer, val_loader, n_batches=n_batches_val)

        history.append({"epoch": epoch, "train_loss": tr_loss,
                         "train_acc": tr_acc, "val_acc": val_acc})
        print(f"    E{epoch+1:3d}/{n_epochs}  loss={tr_loss:.4f}  "
              f"train_acc={tr_acc:.3f}  val_acc={val_acc:.3f}")

        # Checkpoint after every epoch: a session drop loses at most one epoch.
        common.save_checkpoint(
            {"head_state": trainer.head.state_dict(), "opt_state": opt.state_dict(),
             "epoch": epoch, "history": history},
            ckpt_path)

    return trainer, history


@torch.no_grad()
def eval_pipeline(trainer, loader, n_batches=None):
    """Evaluate a trained `AblationTrainer` (head-adapted, CFG['n_classes']-way)."""
    trainer.eval()
    correct = total = total_gc = 0
    for bi, (xb, yb) in enumerate(loader):
        if n_batches is not None and bi >= n_batches: break
        xb, yb = xb.to(device), yb.to(device)
        x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
        logits, gc = trainer(x_raw, return_glance_count=True)
        total_gc += gc.sum().item()
        correct += (logits.argmax(1) == yb).sum().item()
        total   += xb.size(0)
    return correct / max(total, 1), total_gc / max(total, 1)

In [ ]:
# ── Run leave-one-out ablation (checkpointed, resumable per condition) ────
# NOTE: CIFAR-10-C-based corruption-robustness breakdown (noise/blur/weather/
# digital) doesn't apply at ImageNet-100/224px scale -- CIFAR-10-C's precomputed
# arrays are native 32x32. An ImageNet-scale corruption benchmark (ImageNet-C,
# or on-the-fly corruption transforms) would need to be wired in separately;
# for now this loop reports clean accuracy + glance efficiency only.

n_epochs      = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches     = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]
n_batches_val = 10 if CFG["smoke_test"] else None

print("Running leave-one-out ablation sweep"
      + (" (smoke_test)..." if CFG["smoke_test"] else " (real ImageNet-100)..."))

ablation_results = {}
trainers = {}
for lbl, pipeline in ablation_models.items():
    print(f"\n── {lbl} ({n_epochs} epochs) ──────────────────────────────")
    common.set_seed(CFG["seed"])
    ckpt_path = os.path.join(CFG["ckpt_dir"], f"06_{lbl}.pt")
    trainer, history = train_ablation(pipeline, CFG["n_classes"], train_ld, val_ld,
                                       n_batches, n_epochs, ckpt_path,
                                       n_batches_val=n_batches_val)
    trainers[lbl] = trainer

    val_acc, avg_gc = eval_pipeline(trainer, val_ld, n_batches=n_batches_val)
    efficiency = val_acc / max(avg_gc, 1)

    ablation_results[lbl] = {
        "val_acc": val_acc,
        "avg_glances": avg_gc,
        "efficiency": efficiency,
        "history": history,
    }
    print(f"  {lbl:20s}  final: val_acc={val_acc:.4f}  avg_glances={avg_gc:.2f}  "
          f"efficiency={efficiency:.4f}")

print(f"\n{'Variant':20s}  {'Clean acc':>10}  {'Avg glances':>12}  {'Efficiency':>10}")
print("-" * 58)
for lbl, res in ablation_results.items():
    print(f"  {lbl:20s}  {res['val_acc']:10.4f}  {res['avg_glances']:12.2f}  {res['efficiency']:10.4f}")

# Compute marginal contributions relative to full model
full_acc = ablation_results["full"]["val_acc"]
print(f"\n{'Marginal contribution Delta_k = full - ablated':}")
for lbl, res in ablation_results.items():
    if lbl == "full": continue
    delta = full_acc - res["val_acc"]
    print(f"  {lbl:20s}  Delta_k = {delta:+.4f}")

if CFG["smoke_test"]:
    print("\n(smoke_test=True: real ImageNet-100 if present under data/imagenet100/, "
          "else a clearly-labelled synthetic pipeline check -- see datasets.py "
          "honest-fallback policy; few epochs/batches means deltas are noisy. "
          "Set smoke_test=False for real results.)")

# Figure: training curves per condition (loss + train/val accuracy across epochs)
fig_curves, ax_curves = plt.subplots(1, 2, figsize=(12, 4.5))
cmap = plt.cm.tab10
for i, (lbl, res) in enumerate(ablation_results.items()):
    hist = res["history"]
    if not hist: continue
    color = cmap(i / max(len(ablation_results) - 1, 1))
    epochs_h = [h["epoch"] for h in hist]
    ax_curves[0].plot(epochs_h, [h["train_loss"] for h in hist], "-", color=color, label=lbl)
    ax_curves[1].plot(epochs_h, [h["val_acc"] for h in hist], "-", color=color, label=lbl)
ax_curves[0].set_xlabel("epoch"); ax_curves[0].set_ylabel("train loss")
ax_curves[0].set_title("Training loss per epoch"); ax_curves[0].legend(fontsize=7); ax_curves[0].grid(True, alpha=0.3)
ax_curves[1].set_xlabel("epoch"); ax_curves[1].set_ylabel("val accuracy")
ax_curves[1].set_title("Validation accuracy per epoch"); ax_curves[1].legend(fontsize=7); ax_curves[1].grid(True, alpha=0.3)
fig_curves.suptitle("Leave-one-out ablation: training progress per condition", fontsize=11, y=1.03)
plt.tight_layout()
loo_curves_path = os.path.join(CFG["result_dir"], "06_ablation_training_curves.png")
common.save_figure(fig_curves, loo_curves_path, dpi=130)
plt.close(fig_curves)
print(f"Saved: {loo_curves_path}")

In [ ]:
# ── Adversarial robustness for full model ─────────────────────────────────

print("PGD robustness evaluation (full model, EOT)...")

# NOTE: the raw `ablation_models["full"]` pipeline outputs its backbone's native
# class count (e.g. 1000 for ImageNet-pretrained VOneResNet-50), not CIFAR-10's 10 --
# comparing its argmax against CIFAR-10 labels directly would be a category mismatch.
# `trainers["full"]` (built in the leave-one-out cell above) is the same frozen
# pipeline plus the trained CIFAR-10 linear head, so use that instead.
full_trainer = trainers["full"]

# Small batch from val set
xb, yb = next(iter(val_ld))
xb, yb = xb.to(device), yb.to(device)
x_raw  = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)

# PGD against the full trainer (pipeline + CIFAR-10 head)
# EOT-N: needed because C3 (VOneBlock Poisson) makes the forward stochastic
x_adv = pgd_attack(full_trainer, x_raw, yb,
                    eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                    steps=CFG["pgd_steps"], eot_samples=CFG["eot_samples"])

with torch.no_grad():
    clean_logits = full_trainer(x_raw)
    adv_logits   = full_trainer(x_adv)

clean_acc = (clean_logits.argmax(1) == yb).float().mean().item()
adv_acc   = (adv_logits.argmax(1)   == yb).float().mean().item()
print(f"  Clean accuracy  : {clean_acc:.4f}")
print(f"  Robust accuracy : {adv_acc:.4f}  (PGD eps={CFG['eps']:.4f}, EOT={CFG['eot_samples']})")
print("  (smoke_test=True: few epochs on possibly-synthetic data -> not meaningful yet.)"
      if CFG["smoke_test"] else "")

In [ ]:
# ── Ablation table figure ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels_short = [l.replace("no_", "-").replace("_", " ") for l in ABLATION_LABELS]
accs = [ablation_results[l]["val_acc"]     for l in ABLATION_LABELS]
gcs  = [ablation_results[l]["avg_glances"] for l in ABLATION_LABELS]
effs = [ablation_results[l]["efficiency"]  for l in ABLATION_LABELS]

colors = plt.cm.Set2(np.linspace(0, 1, len(ABLATION_LABELS)))

ax = axes[0]
bars = ax.barh(range(len(ABLATION_LABELS)), accs, color=colors)
ax.set_yticks(range(len(ABLATION_LABELS)))
ax.set_yticklabels(labels_short, fontsize=9)
ax.set_xlabel("Val accuracy")
ax.set_title("Leave-one-out ablation: clean accuracy\n(smoke_test — not meaningful)")
ax.axvline(accs[0], color="k", lw=1, ls="--", alpha=0.5, label="full model")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=0.3)

ax = axes[1]
ax.scatter(gcs, accs, c=range(len(ABLATION_LABELS)), cmap="Set2", s=120, zorder=5)
for i, lbl in enumerate(labels_short):
    ax.annotate(lbl, (gcs[i], accs[i]), textcoords="offset points",
                xytext=(5, 3), fontsize=7)
ax.set_xlabel("Avg glances per image")
ax.set_ylabel("Val accuracy")
ax.set_title("Efficiency curve: accuracy vs glances\n(each point = one ablation condition)")
ax.grid(True, alpha=0.3)

fig.suptitle(
    "Full model leave-one-out ablation matrix\n"
    "(smoke_test=True: random weights + synthetic data — for pipeline verification only)",
    fontsize=9, y=1.02,
)
plt.tight_layout()
tbl_path = os.path.join(CFG["result_dir"], "06_ablation_table.png")
common.save_figure(fig, tbl_path, dpi=130)
plt.close(fig)
print(f"Saved: {tbl_path}")


In [ ]:
# ── Beta sweep for the "full" model (VOneNet + trace periphery + multi-glance) ──
# Tests whether the SNR/beta "sweet spot" (docs Ablation A6) still holds once
# VOneNet noise (C3) and IT-feedback multi-glance (C5) are combined with the
# trace-based periphery (C2), now at ImageNet-100/224px (VOneNet's native scale).
# Notebook 07 sweeps beta for a *plain* ResNet-50 + trace periphery on CIFAR-10;
# this repeats that sweep for the full combined pipeline at the matching scale.
# beta=1-3 alone barely change alpha(r) within an 8deg field of view (see
# 02_foveation_rblur_and_periphery.ipynb's beta-sweep figure), hence 5, 8 too.
# Checkpointed per beta (06_full_beta{N}.pt) -- resumable, same as the
# leave-one-out cell above.
#
# NOTE: no CIFAR-10-C-style corruption breakdown here (doesn't apply at this
# scale/resolution -- see note in the leave-one-out cell above) -- clean
# accuracy + PGD robust accuracy only.

n_epochs_beta      = CFG["n_epochs_smoke"]  if CFG["smoke_test"] else CFG["n_epochs_full"]
n_batches_beta      = CFG["n_batches_smoke"] if CFG["smoke_test"] else CFG["n_batches_full"]
n_batches_val_beta  = 10 if CFG["smoke_test"] else None

print("Running beta sweep for the full model (VOneNet + trace periphery + multi-glance)...")

beta_sweep_results = {}
for beta_val in CFG["beta_sweep"]:
    print(f"\n── beta={beta_val:g} ({n_epochs_beta} epochs) ──────────────────────────────")
    common.set_seed(CFG["seed"])
    snr_beta = SNRProfile(snr0_db=CFG["snr0_db"], beta=beta_val, ppd=CFG["ppd"])
    full_pipeline_beta = build_variant("full", PRETRAINED, snr_beta, fixations, CFG).to(device)

    ckpt_path = os.path.join(CFG["ckpt_dir"], f"06_full_beta{beta_val:.0f}.pt")
    trainer_beta, history = train_ablation(full_pipeline_beta, CFG["n_classes"], train_ld, val_ld,
                                            n_batches_beta, n_epochs_beta, ckpt_path,
                                            n_batches_val=n_batches_val_beta)
    val_acc, avg_gc = eval_pipeline(trainer_beta, val_ld, n_batches=n_batches_val_beta)

    xb, yb = next(iter(val_ld))
    xb, yb = xb.to(device), yb.to(device)
    x_raw = (xb * STD.to(device) + MU.to(device)).clamp(0.0, 1.0)
    x_adv = pgd_attack(trainer_beta, x_raw, yb,
                        eps=CFG["eps"], alpha=CFG["pgd_alpha"],
                        steps=CFG["pgd_steps"], eot_samples=CFG["eot_samples"])
    with torch.no_grad():
        robust_acc = (trainer_beta(x_adv).argmax(1) == yb).float().mean().item()

    beta_sweep_results[f"beta{beta_val:.0f}"] = {
        "beta": beta_val, "val_acc": val_acc, "robust_acc": robust_acc, "history": history,
    }
    print(f"  beta={beta_val:g}  final: val_acc={val_acc:.4f}  robust_acc={robust_acc:.4f}")

print(f"\n{'beta':>6}  {'val_acc':>10}  {'robust_acc':>11}")
print("-" * 32)
for key in sorted(beta_sweep_results, key=lambda k: beta_sweep_results[k]["beta"]):
    r = beta_sweep_results[key]
    print(f"{r['beta']:6.1f}  {r['val_acc']:10.4f}  {r['robust_acc']:11.4f}")

if CFG["smoke_test"]:
    print("\n(smoke_test=True: few epochs/batches -> not meaningful yet.)")

# Figure A: beta vs final clean accuracy / PGD robust accuracy
fig3, ax3s = plt.subplots(1, 2, figsize=(11, 4.5))
betas_sorted = sorted(beta_sweep_results, key=lambda k: beta_sweep_results[k]["beta"])
xs = [beta_sweep_results[k]["beta"] for k in betas_sorted]

ax = ax3s[0]
ax.plot(xs, [beta_sweep_results[k]["val_acc"] for k in betas_sorted], "o-", color="steelblue", markersize=8)
ax.set_xlabel("beta"); ax.set_ylabel("val_acc")
ax.set_title("Clean ImageNet-100 accuracy\n(higher = better)", fontsize=9)
ax.grid(True, alpha=0.3)

ax = ax3s[1]
ax.plot(xs, [beta_sweep_results[k]["robust_acc"] for k in betas_sorted], "o-", color="steelblue", markersize=8)
ax.set_xlabel("beta"); ax.set_ylabel("robust_acc")
ax.set_title(f"PGD robust accuracy (eps={CFG['eps']:.3f})\n(higher = better)", fontsize=9)
ax.grid(True, alpha=0.3)

fig3.suptitle("Full model (VOneNet + trace periphery + multi-glance): beta sweep",
              fontsize=11, y=1.03)
plt.tight_layout()
beta_fig_path = os.path.join(CFG["result_dir"], "06_full_model_beta_sweep.png")
common.save_figure(fig3, beta_fig_path, dpi=130)
plt.close(fig3)
print(f"Saved: {beta_fig_path}")

# Figure B: training curves per beta (loss + train/val accuracy across epochs)
fig4, ax4s = plt.subplots(1, 2, figsize=(11, 4.5))
cmap = plt.cm.plasma
for i, key in enumerate(betas_sorted):
    hist = beta_sweep_results[key]["history"]
    if not hist: continue
    color = cmap(i / max(len(betas_sorted) - 1, 1))
    epochs_h = [h["epoch"] for h in hist]
    ax4s[0].plot(epochs_h, [h["train_loss"] for h in hist], "-", color=color,
                 label=f"beta={beta_sweep_results[key]['beta']:g}")
    ax4s[1].plot(epochs_h, [h["val_acc"] for h in hist], "-", color=color,
                 label=f"beta={beta_sweep_results[key]['beta']:g}")
ax4s[0].set_xlabel("epoch"); ax4s[0].set_ylabel("train loss")
ax4s[0].set_title("Training loss per epoch"); ax4s[0].legend(fontsize=7); ax4s[0].grid(True, alpha=0.3)
ax4s[1].set_xlabel("epoch"); ax4s[1].set_ylabel("val accuracy")
ax4s[1].set_title("Validation accuracy per epoch"); ax4s[1].legend(fontsize=7); ax4s[1].grid(True, alpha=0.3)
fig4.suptitle("Full model beta sweep: training progress", fontsize=11, y=1.03)
plt.tight_layout()
beta_curves_path = os.path.join(CFG["result_dir"], "06_full_model_beta_sweep_curves.png")
common.save_figure(fig4, beta_curves_path, dpi=130)
plt.close(fig4)
print(f"Saved: {beta_curves_path}")

In [ ]:
# ── Fetch & filter ImageNet-C to our 100 classes (Zenodo, Hendrycks & Dietterich 2019) ──
# Full ImageNet-C is ~47 GB across all 1000 ImageNet classes (noise.tar 21GB +
# weather.tar 12GB + blur.tar 7GB + digital.tar 7GB) -- far more than we need.
# Same reasoning as the ImageNet-100 fetch cell above: download to local disk
# (fast, reliable), keep only the 100 WordNet-ID folders matching our
# ImageNet-100 subset (~10% the size), then archive just that filtered result
# to Drive as ONE file so future sessions skip the ~47 GB download entirely.

IMAGENET_C_CORRUPTIONS = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "glass_blur", "motion_blur", "zoom_blur",
    "snow", "frost", "fog", "brightness",
    "contrast", "elastic_transform", "pixelate", "jpeg_compression",
]
IMAGENET_C_CATEGORIES = {
    "noise":   ["gaussian_noise", "shot_noise", "impulse_noise"],
    "blur":    ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
    "weather": ["snow", "frost", "fog", "brightness"],
    "digital": ["contrast", "elastic_transform", "pixelate", "jpeg_compression"],
}

IMAGENET_C_LOCAL_ROOT = os.path.join(CFG["imagenet100_data_dir"], "imagenet100-c")
IMAGENET_C_ARCHIVE     = os.path.join(CFG["data_dir"], "imagenet100_c_prepared.tar")

def _imagenet_c_complete(root, corruptions, severities=range(1, 6)):
    if not os.path.isdir(root):
        return False
    for c in corruptions:
        for s in severities:
            d = os.path.join(root, c, str(s))
            if not os.path.isdir(d) or not os.listdir(d):
                return False
    return True

if _imagenet_c_complete(IMAGENET_C_LOCAL_ROOT, IMAGENET_C_CORRUPTIONS):
    print(f"ImageNet-C (filtered) already on local disk: {IMAGENET_C_LOCAL_ROOT}")
elif CFG["smoke_test"]:
    print("smoke_test=True: skipping ImageNet-C fetch "
          "(corruption breakdown below will be NaN until smoke_test=False).")
elif os.path.exists(IMAGENET_C_ARCHIVE):
    import tarfile
    print(f"Extracting cached filtered ImageNet-C archive from Drive "
          f"({os.path.getsize(IMAGENET_C_ARCHIVE) / 1e9:.2f} GB)...")
    os.makedirs(IMAGENET_C_LOCAL_ROOT, exist_ok=True)
    with tarfile.open(IMAGENET_C_ARCHIVE) as tf:
        tf.extractall(IMAGENET_C_LOCAL_ROOT)
    print("Extracted.")
else:
    import shutil, tarfile, urllib.request

    imagenet100_classes = set(os.listdir(IMAGENET100_TRAIN))
    print(f"Filtering ImageNet-C to the {len(imagenet100_classes)} classes in our ImageNet-100 subset.")

    raw_dir = "/content/imagenet_c_raw" if IN_COLAB else os.path.join(CFG["data_dir"], "_imagenet_c_raw")
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(IMAGENET_C_LOCAL_ROOT, exist_ok=True)

    # Category tars -- each extracts directly to its corruption-name subfolders
    # (e.g. blur.tar -> defocus_blur/, glass_blur/, motion_blur/, zoom_blur/),
    # each holding severity/<wnid>/*.JPEG.
    urls = {
        "noise.tar":   "https://zenodo.org/record/2235448/files/noise.tar",
        "blur.tar":    "https://zenodo.org/record/2235448/files/blur.tar",
        "weather.tar": "https://zenodo.org/record/2235448/files/weather.tar",
        "digital.tar": "https://zenodo.org/record/2235448/files/digital.tar",
    }
    for fname, url in urls.items():
        tar_path = os.path.join(raw_dir, fname)
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, tar_path)
        print(f"  Extracting + filtering to {len(imagenet100_classes)} classes...")
        with tarfile.open(tar_path) as tf:
            members = [m for m in tf.getmembers()
                       if m.isfile() and m.name.split("/")[-2] in imagenet100_classes]
            tf.extractall(IMAGENET_C_LOCAL_ROOT, members=members)
        os.remove(tar_path)   # free local disk immediately, one tar at a time

    shutil.rmtree(raw_dir, ignore_errors=True)
    n_files = sum(len(files) for _, _, files in os.walk(IMAGENET_C_LOCAL_ROOT))
    print(f"ImageNet-C (filtered) ready on local disk: {n_files} files.")

    print(f"Archiving filtered ImageNet-C to Drive: {IMAGENET_C_ARCHIVE} ...")
    os.makedirs(os.path.dirname(IMAGENET_C_ARCHIVE), exist_ok=True)
    with tarfile.open(IMAGENET_C_ARCHIVE, "w") as tf:
        tf.add(IMAGENET_C_LOCAL_ROOT, arcname=".")
    print("Archived.")

In [ ]:
# -- ImageNet-C corruption breakdown: all leave-one-out conditions + all beta values --
# Per-corruption-type AND per-category (noise/blur/weather/digital) error rates,
# not just one averaged number -- reuses each condition's/beta's already-trained
# checkpoint (no retraining here, so this cell can be re-run independently).

_imagenet_c_transform = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def eval_imagenet_c_breakdown(trainer, root, n_batches=None, severities=(1, 3, 5)):
    """Per-corruption-type and per-category top-1 error rates on the (100-class-
    filtered) ImageNet-C set. NaN throughout if ImageNet-C is not present (honest
    fallback, no crash)."""
    per_corruption = {}
    for corruption in IMAGENET_C_CORRUPTIONS:
        errs = []
        for severity in severities:
            split_dir = os.path.join(root, corruption, str(severity))
            if not os.path.isdir(split_dir):
                errs = None
                break
            ds = torchvision.datasets.ImageFolder(root=split_dir, transform=_imagenet_c_transform)
            ld = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False)
            acc, _ = eval_pipeline(trainer, ld, n_batches=n_batches)
            errs.append(1.0 - acc)
        per_corruption[corruption] = float(np.mean(errs)) if errs else float("nan")

    per_category = {}
    for cat, corruptions in IMAGENET_C_CATEGORIES.items():
        vals = [per_corruption[c] for c in corruptions]
        per_category[cat] = float(np.nanmean(vals)) if not all(np.isnan(v) for v in vals) else float("nan")

    overall_vals = list(per_corruption.values())
    overall = (float(np.nanmean(overall_vals))
               if not all(np.isnan(v) for v in overall_vals) else float("nan"))

    return {"per_corruption": per_corruption, "per_category": per_category, "overall": overall}


def print_imagenet_c_breakdown(label, breakdown):
    cat_header = "Category"
    err_header = "Error"
    print("\n  " + label + " -- ImageNet-C breakdown")
    print(f"    {cat_header:<10} {err_header:>8}")
    for cat, err in breakdown["per_category"].items():
        print(f"    {cat:<10} {err:>8.4f}")
    print("    " + "-" * 19)
    overall_label = "Overall"
    print(f"    {overall_label:<10} {breakdown['overall']:>8.4f}")
    print("    Per-corruption-type:")
    for cat, corruptions in IMAGENET_C_CATEGORIES.items():
        for c in corruptions:
            err = breakdown["per_corruption"][c]
            print(f"      [{cat[:4]}] {c:<20} {err:>8.4f}")


n_batches_imagenet_c = CFG["n_batches_imagenet_c_smoke"] if CFG["smoke_test"] else None
imagenet_c_severities = (3,) if CFG["smoke_test"] else (1, 3, 5)

# -- Leave-one-out conditions --
print("Evaluating leave-one-out conditions on ImageNet-C...")
corruption_c_results = {}
for lbl, pipeline in ablation_models.items():
    ckpt_path = os.path.join(CFG["ckpt_dir"], f"06_{lbl}.pt")
    ckpt = common.load_checkpoint(ckpt_path, map_location=device)
    if ckpt is None:
        print(f"  {lbl}: no checkpoint found -- run the leave-one-out cell first. Skipping.")
        continue
    trainer = AblationTrainer(pipeline, CFG["n_classes"], CFG["image_size"]).to(device)
    trainer.head.load_state_dict(ckpt["head_state"])
    trainer.eval()

    breakdown = eval_imagenet_c_breakdown(trainer, IMAGENET_C_LOCAL_ROOT,
                                           n_batches=n_batches_imagenet_c,
                                           severities=imagenet_c_severities)
    corruption_c_results[lbl] = breakdown
    print_imagenet_c_breakdown(lbl, breakdown)

# -- Beta-sweep conditions --
print("\n\nEvaluating beta-sweep conditions on ImageNet-C...")
corruption_c_beta_results = {}
for beta_val in CFG["beta_sweep"]:
    ckpt_path = os.path.join(CFG["ckpt_dir"], f"06_full_beta{beta_val:.0f}.pt")
    ckpt = common.load_checkpoint(ckpt_path, map_location=device)
    if ckpt is None:
        print(f"  beta={beta_val:g}: no checkpoint found -- run the beta-sweep cell first. Skipping.")
        continue
    snr_beta = SNRProfile(snr0_db=CFG["snr0_db"], beta=beta_val, ppd=CFG["ppd"])
    full_pipeline_beta = build_variant("full", PRETRAINED, snr_beta, fixations, CFG).to(device)
    trainer_beta = AblationTrainer(full_pipeline_beta, CFG["n_classes"], CFG["image_size"]).to(device)
    trainer_beta.head.load_state_dict(ckpt["head_state"])
    trainer_beta.eval()

    breakdown = eval_imagenet_c_breakdown(trainer_beta, IMAGENET_C_LOCAL_ROOT,
                                           n_batches=n_batches_imagenet_c,
                                           severities=imagenet_c_severities)
    corruption_c_beta_results[f"beta{beta_val:.0f}"] = {"beta": beta_val, "breakdown": breakdown}
    print_imagenet_c_breakdown(f"beta={beta_val:g}", breakdown)

if CFG["smoke_test"]:
    print("\n(smoke_test=True: breakdown is NaN until ImageNet-C is fetched -- "
          "set CFG['smoke_test']=False and re-run the fetch cell above.)")

# -- Figure A: per-category error, grouped by leave-one-out condition --
if corruption_c_results:
    _cond_labels = list(corruption_c_results.keys())
    _categories  = list(IMAGENET_C_CATEGORIES.keys())
    _x = np.arange(len(_cond_labels))
    _width = 0.8 / len(_categories)

    fig_c, ax_c = plt.subplots(figsize=(10, 5))
    for ci, cat in enumerate(_categories):
        vals = [corruption_c_results[lbl]["per_category"][cat] for lbl in _cond_labels]
        ax_c.bar(_x + ci * _width, vals, _width, label=cat)
    ax_c.set_xticks(_x + _width * (len(_categories) - 1) / 2)
    ax_c.set_xticklabels([l.replace("_", " ") for l in _cond_labels], rotation=20, ha="right")
    ax_c.set_ylabel("Top-1 error rate")
    ax_c.set_title("ImageNet-C error by category, per leave-one-out condition")
    ax_c.legend(title="corruption category", fontsize=8)
    ax_c.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    c_cat_path = os.path.join(CFG["result_dir"], "06_ablation_imagenet_c_by_category.png")
    common.save_figure(fig_c, c_cat_path, dpi=130)
    plt.close(fig_c)
    print(f"\nSaved: {c_cat_path}")

# -- Figure B: beta vs per-category + overall ImageNet-C error --
if corruption_c_beta_results:
    betas_c_sorted = sorted(corruption_c_beta_results,
                             key=lambda k: corruption_c_beta_results[k]["beta"])
    xs_c = [corruption_c_beta_results[k]["beta"] for k in betas_c_sorted]

    fig_bc, ax_bc = plt.subplots(figsize=(7, 5))
    for cat in IMAGENET_C_CATEGORIES:
        ys = [corruption_c_beta_results[k]["breakdown"]["per_category"][cat] for k in betas_c_sorted]
        ax_bc.plot(xs_c, ys, "o--", markersize=5, alpha=0.7, label=cat)
    ax_bc.plot(xs_c, [corruption_c_beta_results[k]["breakdown"]["overall"] for k in betas_c_sorted],
               "o-", color="black", markersize=8, linewidth=2, label="overall")
    ax_bc.set_xlabel("beta"); ax_bc.set_ylabel("error rate")
    ax_bc.set_title("Full model beta sweep: ImageNet-C error by category\n(lower = better)")
    ax_bc.legend(fontsize=8); ax_bc.grid(True, alpha=0.3)
    plt.tight_layout()
    beta_c_path = os.path.join(CFG["result_dir"], "06_full_model_beta_sweep_imagenet_c.png")
    common.save_figure(fig_bc, beta_c_path, dpi=130)
    plt.close(fig_bc)
    print(f"Saved: {beta_c_path}")

In [ ]:
# ── Save results ─────────────────────────────────────────────────────────
summary = {
    "notebook": "06_full_model_and_ablations",
    "cfg":      {k: v for k, v in CFG.items() if not k.endswith("_dir")},
    "ablation_table": ablation_results,
    "adversarial": {
        "clean_acc": clean_acc, "robust_acc": adv_acc,
        "eps": CFG["eps"], "eot_samples": CFG["eot_samples"],
    },
    "full_model_beta_sweep": beta_sweep_results,
    "imagenet_c_by_condition": corruption_c_results,
    "imagenet_c_by_beta": corruption_c_beta_results,
    "note": "smoke_test=True: all numbers reflect pipeline, not real performance.",
}
rpath = os.path.join(CFG["result_dir"], "06_ablation_table.json")
common.save_json(summary, rpath)
common.save_config(CFG, os.path.join(CFG["result_dir"], "06_config.json"))

print(f"Results: {rpath}")
print(f"06_ablation_table.png : {os.path.exists(tbl_path)}")
print(f"06_ablation_training_curves.png : {os.path.exists(loo_curves_path)}")
print(f"06_full_model_beta_sweep.png : {os.path.exists(beta_fig_path)}")
print(f"06_full_model_beta_sweep_curves.png : {os.path.exists(beta_curves_path)}")
print(f"06_ablation_imagenet_c_by_category.png : {os.path.exists(os.path.join(CFG['result_dir'], '06_ablation_imagenet_c_by_category.png'))}")
print(f"06_full_model_beta_sweep_imagenet_c.png : {os.path.exists(os.path.join(CFG['result_dir'], '06_full_model_beta_sweep_imagenet_c.png'))}")
print(f"Checkpoints: {CFG['ckpt_dir']} (per-condition, resumable)")
print()
print("=" * 60)
print("ALL NOTEBOOKS COMPLETE")
print("=" * 60)
nbs = ["00_setup_and_data", "01_baseline_reproduce",
       "02_foveation_rblur_and_periphery", "03_v1_block",
       "04_it_feedback_multiglance", "05_mftma_certification",
       "06_full_model_and_ablations"]
for nb in nbs:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "notebooks", nb + ".ipynb"))
    print(f"  {'OK' if exists else 'MISSING':6s} notebooks/{nb}.ipynb")
print()
src_files = ["common.py", "datasets.py", "eval_harness.py", "models.py",
             "overrides.py", "foveation.py", "it_feedback.py", "mftma.py"]
for sf in src_files:
    exists = os.path.exists(os.path.join(PROJECT_ROOT, "src", sf))
    print(f"  {'OK' if exists else 'MISSING':6s} src/{sf}")
